# 08.07 · Long-term memory store inside graphs

**Module 08 · LangGraph Production Features** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lesson 03.04 (long-term memory with stores); Lessons 08.01–08.03. |
| 🔑 Keys | none |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Use** a store from graph nodes: `compile(store=...)` and `runtime.store`, with the user from `runtime.context`.
2. **Design** namespaces (per user, per app, per org) and what goes in the store vs the checkpoint.
3. **Search** memories by meaning (semantic index) and by filter, and let a node **write** memories.
4. **Verify** a store with a contract test before you swap `InMemoryStore` for SQLite or Postgres.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Stores](https://docs.langchain.com/oss/python/langgraph/stores) | `BaseStore`: `put`/`get`/`search`/`delete`, namespaces, semantic search, `runtime.store` |
| [Add memory](https://docs.langchain.com/oss/python/langgraph/add-memory) | Long-term memory in graphs, `context_schema`, user ids |
| [Persistence: memory store](https://docs.langchain.com/oss/python/langgraph/persistence) | Checkpointer (one thread) vs store (across threads) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langgraph-checkpoint-sqlite",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/08-langgraph-production-features/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 08 can reuse Module 06): no usage telemetry.
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

> Everything is 🧪 offline. Semantic search uses `mastery.memory.HashingEmbeddings` (from Module 03): deterministic word-overlap vectors, no download. Swap in real embeddings (Module 06's fastembed) for meaning-level matches.

---
## Part 1 · Checkpointer vs store

| | Checkpointer (Lessons 08.01–08.04) | Store (this lesson) |
|---|---|---|
| Scope | One **thread** | Across **all** threads |
| Holds | The graph's state at each step | Items you choose: `(namespace, key) → dict` |
| Written by | LangGraph, automatically | **Your** nodes (`runtime.store.put`) |
| Example | "The draft we're reviewing right now" | "Ana prefers one short paragraph" |

Pass both at compile time. Nodes reach the store through `runtime.store`, and **who** is calling through `runtime.context` (a `context_schema`, Lesson 03.04).

In [ ]:
# 🧪 Offline: a graph that reads and writes per-user preferences.
from dataclasses import dataclass

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph
from langgraph.runtime import Runtime
from mastery.memory import HashingEmbeddings, make_store
from typing_extensions import TypedDict


@dataclass
class Ctx:
    user_id: str


class Chat(TypedDict, total=False):
    message: str
    reply: str


def remember(state: Chat, runtime: Runtime[Ctx]) -> dict:
    """Store a preference when the user states one ("I prefer ...")."""
    text = state["message"]
    if text.lower().startswith("i prefer"):
        ns = ("preferences", runtime.context.user_id)
        key = f"pref-{len(runtime.store.search(ns)) + 1}"
        runtime.store.put(ns, key, {"text": text[len("i prefer"):].strip(" .")})
    return {}


def respond(state: Chat, runtime: Runtime[Ctx]) -> dict:
    ns = ("preferences", runtime.context.user_id)
    prefs = [item.value["text"] for item in runtime.store.search(ns, query=state["message"], limit=2)]
    return {"reply": f"(using preferences: {prefs or 'none'}) Answer to: {state['message']}"}


store = make_store("memory", embed=HashingEmbeddings())        # semantic index on the whole value (fields=["$"]: metadata like "topic" is embedded too; fields=["text"] is cleaner)
assistant = (StateGraph(Chat, context_schema=Ctx)
             .add_node("remember", remember).add_node("respond", respond)
             .add_edge(START, "remember").add_edge("remember", "respond")
             .compile(checkpointer=InMemorySaver(), store=store))


def say(user, thread, message):
    out = assistant.invoke({"message": message}, {"configurable": {"thread_id": thread}}, context=Ctx(user_id=user))
    print(f"{user}/{thread}: {out['reply']}")


say("ana", "t1", "I prefer short answers with one example.")
say("ana", "t2", "How should I store the battery?")          # a NEW thread: the checkpoint is empty, the store isn't
say("ben", "t3", "How should I store the battery?")          # another user: another namespace

**You should now see** three replies: the first already uses Ana's new preference (`remember` ran before `respond`); the second, on a **new thread**, still finds `short answers with one example`; Ben's says `using preferences: none`.

### What just happened?

The checkpointer gave each thread its own state; the **store** carried Ana's preference across threads. The namespace `("preferences", user_id)` keeps users apart: a node only sees what it asks for. `search(ns, query=...)` ranked Ana's items by similarity (with only one item, it's simply returned).

---
## Part 2 · Namespaces and search

In [ ]:
# 🧪 Offline: more memories, three ways to search.
store.put(("preferences", "ana"), "pref-2", {"text": "metric units, never imperial", "topic": "units"})
store.put(("preferences", "ana"), "pref-3", {"text": "friendly tone, no jargon", "topic": "tone"})
store.put(("facts", "ana"), "station", {"text": "owns a Nimbus Pro mounted on a balcony", "topic": "device"})

print("one namespace  :", sorted(i.key for i in store.search(("preferences", "ana"))))
print("by filter      :", [i.key for i in store.search(("preferences", "ana"), filter={"topic": "units"})])
print("by meaning     :", [i.key for i in store.search(("preferences", "ana"), query="what units should I use?", limit=1)])
print("namespaces     :", sorted(store.list_namespaces(prefix=("preferences",))))
print("everything ana :", sorted((tuple(i.namespace)[0], i.key) for p in (("preferences", "ana"), ("facts", "ana")) for i in store.search(p)))

**You should now see** `one namespace  : ['pref-1', 'pref-2', 'pref-3']`, `by filter      : ['pref-2']`, `by meaning     : ['pref-2']` (it shares the word "units"), `namespaces` with Ana's and nothing for Ben (he never stated a preference), and Ana's four items across both namespaces.

Namespace design is a product decision:

| Namespace | For |
|---|---|
| `("preferences", user_id)` | How a user wants answers (Project 08's stretch goal) |
| `("facts", user_id)` | What you know about the user (devices, plan, language) |
| `("org", org_id, "policies")` | Shared knowledge for a whole team |
| `("app", "prompts")` | Configuration you can change without deploying |

Prefix search (`search(("preferences",))`) reads **across** users: fine for admin tools, never in a user-facing node.

---
## Part 3 · The Project 08 agent with preferences

`build_research_graph(..., store=store)` reads `("preferences", user_id)` in the `draft` node; `user_id` comes from `ResearchContext`. The offline writer understands "paragraph" (one paragraph instead of bullets).

In [ ]:
# 🧪 Offline: same question, two users, different style.
from mastery.graphs.hitl import adrive, scripted_reviewer
from mastery.graphs.research import ResearchContext, build_research_graph, corpus_docs_search, offline_models, offline_web_search
from mastery.graphs.state import memory_checkpointer

prefs = make_store("memory")
prefs.put(("preferences", "ana"), "style", {"text": "one paragraph, no bullet lists"})
agent = build_research_graph(offline_models(), web_search=offline_web_search(), docs_search=corpus_docs_search(),
                             checkpointer=memory_checkpointer(), store=prefs)

for user in ("ana", "ben"):
    out = await adrive(agent, {"question": "How do I keep the battery healthy in winter?"},
                       {"configurable": {"thread_id": f"prefs-{user}"}}, scripted_reviewer(["approve"]),
                       context=ResearchContext(user_id=user))
    body = out["values"]["final"].split("\n\n")[1]
    print(f"{user}: {'bullets' if body.startswith('- ') else 'paragraph'} -> {body[:110]}...")

**You should now see** `ana: paragraph -> ...` and `ben: bullets -> - ...`: one graph, one store, different output per user.

> With live models the preference text goes into the writer's prompt (`Writing preferences: ...`), so any instruction works, not only "paragraph". In `langgraph dev` the server supplies its own store: pass `context={"user_id": "ana"}` with the run (Project 08 README). Fine locally; in a real app the `user_id` must come from authentication, never from the client (Module 12).

---
## Part 4 · A contract test for stores

Graph code only calls `put`, `get`, `search`, `delete`, `list_namespaces`. If another store behaves the same way for those calls, you can swap it in. `mastery.graphs.stores.check_store_contract(store)` runs 10 checks (11 with `semantic=True`) and cleans up after itself.

In [ ]:
# 🧪 Offline: the same contract on InMemoryStore and SqliteStore, with and without a vector index.
from mastery.graphs.stores import check_store_contract
from mastery.memory import close

OUT = REPO_ROOT / "data" / "outputs" / "m08"
OUT.mkdir(parents=True, exist_ok=True)
candidates = {
    "memory": make_store("memory"),
    "memory+vectors": make_store("memory", embed=HashingEmbeddings()),
    "sqlite": make_store("sqlite", OUT / "08-07-store.sqlite"),
    "sqlite+vectors": make_store("sqlite", OUT / "08-07-store-vec.sqlite", embed=HashingEmbeddings()),
}
for name, candidate in candidates.items():
    results = check_store_contract(candidate, semantic=name.endswith("vectors"))
    failed = [check for check, ok in results.items() if not ok]
    print(f"{name:<15} {sum(results.values())}/{len(results)} passed", failed or "")
    close(candidate)
print("checks:", list(results))

**You should now see** `memory 10/10`, `memory+vectors 11/11`, `sqlite 10/10`, `sqlite+vectors 11/11` and the list of check names (get, overwrite, filter, prefix, `list_namespaces`, semantic, delete, ...).

Run the same function against a Postgres store (`langgraph-checkpoint-postgres`), a Redis store or your own `BaseStore` subclass before you switch. A failing check tells you exactly which behaviour your nodes would lose.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| `runtime.store` is `None` | Compiled without `store=` | `compile(store=...)` (the server provides one) |
| `runtime.context` is `None` / missing `user_id` | No `context=` in `invoke` | `invoke(..., context=Ctx(user_id=...))` |
| `search(query=...)` returns items in insertion order | No vector index on the store | `make_store(..., embed=...)` |
| Users see each other's memories | Namespace without the user id, or prefix search in a node | Always include the user id in the namespace (`(kind, user_id)` here, `("learners", user_id, ...)` in Module 03); never search a prefix above the user id in user-facing code |
| `PermissionError` deleting `store.sqlite` | Store still open | `close(store)` |
| Memories grow forever | No expiry | SQLite `ttl={"default_ttl": minutes}` + `store.sweep_ttl()`, or delete on a schedule |

## ⚠️ Production notes

> ⚠️ **Production note: memories are user data.**
> A store holds what users told you, across conversations. Give users a way to see and delete it (`search` + `delete` per namespace), don't store secrets or payment data, and apply the same guardrails as for inputs (Module 04): a stored "preference" is also a prompt-injection vector.

> ⚠️ **Production note: write memories deliberately.**
> Writing in every turn fills the store with noise. Decide **which** events create a memory (an explicit "I prefer…", a confirmed fact), store small structured values, and let a background job summarize or expire old items (Lesson 03.05 compares hot-path and background writes; the course doesn't build the background job).

## 🏋️ Exercises

1. **(Easy)** In Part 1, ask Ben to say "I prefer bullet points." and then ask the battery question. What does his reply use?
2. **(Medium)** Add a `forget` path: when the message is "forget my preferences", delete every item in the user's `preferences` namespace.
3. **(Medium)** Write a memory **with a namespace per org**: `("org", "acme", "policies")`, and read it in `respond` next to the user's preferences.
4. **(Stretch)** Write your own tiny store check: does `store.search(ns, limit=1)` return the **newest** item? Run it on all four stores from Part 4 and log the answer.

<details>
<summary><b>Show solutions</b></summary>

**1.** `(using preferences: ['bullet points']) ...`: his own namespace now has one item.

**2.**
```python
def remember(state, runtime):
    ns = ("preferences", runtime.context.user_id)
    if state["message"].lower().startswith("forget my preferences"):
        for item in runtime.store.search(ns, limit=100):
            runtime.store.delete(ns, item.key)
        return {}
    ...
```

**3.**
```python
store.put(("org", "acme", "policies"), "units", {"text": "always metric"})
org = [i.value["text"] for i in runtime.store.search(("org", "acme", "policies"))]
```
(In a real app, put `org_id` in the context too.)

**4.** Don't rely on it: ordering without a query isn't part of the documented contract. If you need "newest", store a timestamp in the value and sort, or use one fixed key per slot (`put` overwrites).
</details>

## ✅ Checkpoint

- [ ] I used `runtime.store` and `runtime.context` inside nodes.
- [ ] I can explain checkpointer vs store and design a namespace.
- [ ] I searched by namespace, filter and meaning.
- [ ] I ran the store contract test on InMemory and SQLite stores.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Docs drill (5 min)

`inspect.signature(InMemoryStore.search)`: what's the default `limit`? What happens when a namespace holds more items than that and you call `search` without `limit=`?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Signature: `search(namespace_prefix, /, *, query=None, filter=None, limit=10, offset=0, refresh_ttl=None)`.
2. Answer: you silently get the first **10** items; no warning. Put 12 memories in a namespace and `len(store.search(ns))` returns 10.
3. Fix: pass `limit=` (and page with `offset=`) wherever "all items" matters, and check every `search` call in this lesson.

</details>

## 🔎 Research checkpoint

Read [Stores](https://docs.langchain.com/oss/python/langgraph/stores) and the [Backward compatibility](https://docs.langchain.com/oss/python/langgraph/backward-compatibility) page (this module's research checkpoint) and note in [`docs/research-log.md`](../../docs/research-log.md): which store backends exist today, how TTL works on each, and **what happens to in-flight runs (threads paused at an interrupt) when you change a graph**: adding a node, renaming one, changing state keys.

## Summary

- `compile(store=...)`; nodes use `runtime.store` and `runtime.context` (who's asking).
- Checkpointer = one thread's state; store = memories across threads, under namespaces like `("preferences", user_id)`.
- `search(ns)`, `search(ns, filter=...)`, `search(ns, query=...)` (needs an index).
- A contract test makes stores swappable; memories are user data.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria).

[Module 08 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)